In [1]:
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from scipy import sparse
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity, rbf_kernel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Gene Exp

In [2]:
cell_info = pd.read_excel(
    "/Users/inouey2/Downloads/Cell_Lines_Details.xlsx",
    usecols=["Sample Name", "COSMIC identifier"],
)
cell_info.head()

/Users/inouey2/miniconda3/envs/torch/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


,Sample Name,COSMIC identifier
0,A253,906794.0
1,BB30-HNC,753531.0
2,BB49-HNC,753532.0
3,BHY,753535.0
4,BICR10,1290724.0


In [3]:
gene_exp = pd.read_table("/Users/inouey2/Downloads/Cell_line_RMA_proc_basalExp.txt.zip")
gene_exp.index = list(gene_exp["GENE_SYMBOLS"])
gene_exp = gene_exp[~gene_exp.index.isna()]
gene_exp = gene_exp.drop(["GENE_SYMBOLS", "GENE_title"], axis=1)
gene_exp.columns = [float(i[5:]) for i in gene_exp.columns]
gene_exp = gene_exp[sorted(set(gene_exp.columns) & set(cell_info["COSMIC identifier"]))]
gene_exp.columns = [
    cell_info[cell_info["COSMIC identifier"] == i]["Sample Name"].iloc[0]
    for i in gene_exp.columns
]
gene_exp = gene_exp.T
gene_exp.head()

,TSPAN6,TNMD,DPM1,SCYL3,C1orf112,FGR,CFH,FUCA2,GCLC,NFYA,...,LINC00514,OR1D5,ZNF234,MYH4,LINC00526,PPY2,KRT18P55,POLRMTP1,UBL5P2,TBC1D3P5
MC-CAR,3.238273,2.982254,10.235491,4.856061,4.078870,9.116236,3.658590,6.145475,5.042464,5.438402,...,3.103752,3.724013,3.981948,2.823245,5.866047,3.095716,3.274367,3.056214,9.446305,3.530871
PFSK-1,7.780713,2.753253,9.960137,4.351073,3.716740,3.222277,8.221606,3.823474,4.756228,5.805642,...,3.175476,3.779354,4.504481,2.690651,3.347520,3.230713,3.102091,3.169188,9.810430,3.266915
A673,7.301344,2.890533,9.922489,4.125088,3.678987,3.096576,3.588391,4.809305,4.951782,5.089165,...,3.299300,3.762301,4.177345,2.499803,5.054260,3.003521,3.068187,3.135479,9.073222,3.098364
ES3,8.690198,3.091473,9.992487,4.572198,3.333385,3.320793,3.159487,3.515105,5.446361,5.348338,...,3.251885,2.929491,4.702208,2.489674,5.097089,3.114744,3.119647,3.194925,9.013220,3.074187
ES5,8.233101,2.824687,10.015884,4.749715,3.839433,3.142754,5.329830,3.272124,5.538055,6.428482,...,3.081750,3.226083,4.666295,2.491254,6.261573,3.031862,3.322455,2.813440,8.893197,3.266184


# Drug Response

In [4]:
drug_response = pd.read_excel(
    "/Users/inouey2/Downloads/GDSC1_fitted_dose_response_27Oct23.xlsx",
    usecols=["DRUG_NAME", "CELL_LINE_NAME", "Z_SCORE"],
)
drug_response

,CELL_LINE_NAME,DRUG_NAME,Z_SCORE
0,ES5,Erlotinib,1.299144
1,ES7,Erlotinib,0.156076
2,EW-11,Erlotinib,-0.035912
3,SK-ES-1,Erlotinib,-0.434437
4,COLO-829,Erlotinib,0.401702
...,...,...,...
333156,SNU-1040,I-CBP112,0.860626
333157,SNU-61,I-CBP112,1.785602
333158,SNU-81,I-CBP112,0.637308
333159,SNU-C5,I-CBP112,0.089683


In [5]:
l = pd.read_csv("../Figs/nsc_cid_smiles_class_name.csv", index_col=0)[
    ["NAME", "SMILES"]
]
l

,NAME,SMILES
0,p-Toluquinone,CC1=CC(=O)C=CC1=O
1,4-Amino-3-pentadecylphenol,CCCCCCCCCCCCCCCC1=C(C=CC(=C1)O)N
2,3-(Dimethylamino)propiophenone hydrochloride,CN(C)CCC(=O)C1=CC=CC=C1.Cl
3,Cycloheximide,CC1CC(C(=O)C(C1)C(CC2CC(=O)NC(=O)C2)O)C
4,Cycloheximide,CC1CC(C(=O)C(C1)C(CC2CC(=O)NC(=O)C2)O)C
...,...,...
23186,"N-(5-piperazin-1-ylpyridin-2-yl)-4-(7-thia-9,1...",C1CC2=C(C1)SC3=NC=NC(=C23)NC4=C(NN=C4)C(=O)NC5...
23187,"(11Z)-3,5-diethyl-11-(2-oxo-1-prop-2-enylindol...",CCN1C2C(N(C1=O)CC)N3C(=O)C(=C4C5=CC=CC=C5N(C4=...
23188,2-[(E)-benzylideneamino]-4-phenyl-4H-benzo[h]c...,C1=CC=C(C=C1)C=NC2=C(C(C3=C(O2)C4=CC=CC=C4C=C3...
23189,Naporafenib,CC1=C(C=C(C=C1)NC(=O)C2=CC(=NC=C2)C(F)(F)F)C3=...


In [6]:
df = drug_response[drug_response["DRUG_NAME"].isin(l["NAME"])]
df = df[
    df["CELL_LINE_NAME"].isin(set(drug_response.CELL_LINE_NAME) & set(gene_exp.index))
]
df

,CELL_LINE_NAME,DRUG_NAME,Z_SCORE
750,ES5,Sunitinib,0.800508
751,ES7,Sunitinib,0.327295
752,EW-11,Sunitinib,0.695711
753,SK-ES-1,Sunitinib,-0.979122
754,COLO-829,Sunitinib,1.794693
...,...,...,...
330799,SNU-1040,Pictilisib,0.998393
330800,SNU-407,Pictilisib,0.222536
330801,SNU-61,Pictilisib,1.982226
330802,SNU-81,Pictilisib,-0.560742


# Zero shot prediction

In [7]:
unique_drugs = df["DRUG_NAME"].unique()
unique_cells = df["CELL_LINE_NAME"].unique()

# Split drugs and cell lines into training, validation, and test sets
train_drugs, test_val_drugs = train_test_split(
    unique_drugs, test_size=0.5, random_state=42
)
val_drugs, test_drugs = train_test_split(test_val_drugs, test_size=0.5, random_state=42)

train_cells, test_val_cells = train_test_split(
    unique_cells, test_size=0.55, random_state=42
)
val_cells, test_cells = train_test_split(test_val_cells, test_size=0.5, random_state=42)

# Split the dataset
train_df = df[
    (df["DRUG_NAME"].isin(train_drugs)) & (df["CELL_LINE_NAME"].isin(train_cells))
]
val_df = df[(df["DRUG_NAME"].isin(val_drugs)) & (df["CELL_LINE_NAME"].isin(val_cells))]
test_df = df[
    (df["DRUG_NAME"].isin(test_drugs)) & (df["CELL_LINE_NAME"].isin(test_cells))
]


# Function to balance label distribution
def balance_labels(df, threshold=0.5):
    positive = df[df["Z_SCORE"] > 0]
    negative = df[df["Z_SCORE"] <= 0]
    min_count = min(len(positive), len(negative))
    balanced_positive = positive.sample(min_count, random_state=42)
    balanced_negative = negative.sample(min_count, random_state=42)
    return pd.concat([balanced_positive, balanced_negative])


# Balance label distribution across all sets
train_df = balance_labels(train_df)
val_df = balance_labels(val_df)
test_df = balance_labels(test_df)

# Separate features and labels
X_train = train_df[["DRUG_NAME", "CELL_LINE_NAME"]]
y_train = np.array(train_df["Z_SCORE"] > 0, dtype=float)

X_val = val_df[["DRUG_NAME", "CELL_LINE_NAME"]]
y_val = np.array(val_df["Z_SCORE"] > 0, dtype=float)

X_test = test_df[["DRUG_NAME", "CELL_LINE_NAME"]]
y_test = np.array(test_df["Z_SCORE"] > 0, dtype=float)

# Calculate total samples
total_samples = len(X_train) + len(X_val) + len(X_test)


# Function to calculate and format label ratios
def get_label_ratio(y):
    unique, counts = np.unique(y, return_counts=True)
    total = len(y)
    ratio_str = ", ".join(
        [f"Label {label}: {count/total:.2%}" for label, count in zip(unique, counts)]
    )
    return ratio_str


# Display results with percentages and label ratios
print("Train:")
print(X_train.shape, y_train.shape)
print(f"Percentage: {len(X_train)/total_samples:.2%}")
print(f"Label Ratio: {get_label_ratio(y_train)}")

print("\nValidation:")
print(X_val.shape, y_val.shape)
print(f"Percentage: {len(X_val)/total_samples:.2%}")
print(f"Label Ratio: {get_label_ratio(y_val)}")

print("\nTest:")
print(X_test.shape, y_test.shape)
print(f"Percentage: {len(X_test)/total_samples:.2%}")
print(f"Label Ratio: {get_label_ratio(y_test)}")

print(f"\nTotal samples: {total_samples}")
print(
    f"Ratio (Train:Validation:Test): {len(X_train):.0f}:{len(X_val):.0f}:{len(X_test):.0f}"
)
print(
    f"Overall Label Ratio: {get_label_ratio(np.concatenate([y_train, y_val, y_test]))}"
)

X_train.to_csv("../GDSC1_data/train.csv", index=False)
X_test.to_csv("../GDSC1_data/test.csv", index=False)
X_val.to_csv("../GDSC1_data/val.csv", index=False)

np.save("../GDSC1_data/train_labels.npy", y_train)
np.save("../GDSC1_data/test_labels.npy", y_test)
np.save("../GDSC1_data/val_labels.npy", y_val)

Train:
(13546, 2) (13546,)
Percentage: 63.32%
Label Ratio: Label 0.0: 50.00%, Label 1.0: 50.00%

Validation:
(4172, 2) (4172,)
Percentage: 19.50%
Label Ratio: Label 0.0: 50.00%, Label 1.0: 50.00%

Test:
(3674, 2) (3674,)
Percentage: 17.17%
Label Ratio: Label 0.0: 50.00%, Label 1.0: 50.00%

Total samples: 21392
Ratio (Train:Validation:Test): 13546:4172:3674
Overall Label Ratio: Label 0.0: 50.00%, Label 1.0: 50.00%


In [8]:
drug_response = df.pivot_table(
    columns="CELL_LINE_NAME", index="DRUG_NAME", values="Z_SCORE"
)
drug_response.index = list(drug_response.index)
drug_response.columns = list(drug_response.columns)
drug_response = drug_response.fillna(0)
drug_response

,22RV1,23132-87,42-MG-BA,451Lu,639-V,647-V,769-P,786-0,8-MG-BA,8305C,...,WSU-DLCL2,WSU-NHL,YAPC,YH-13,YKG-1,YT,ZR-75-30,huH-1,no-10,no-11
Afatinib,0.648770,0.498498,0.638172,0.629175,-0.009590,-0.267917,-1.221836,-0.275344,0.244164,0.577540,...,0.329646,0.377467,-0.301358,0.156829,0.038681,0.404893,-2.136028,0.908394,0.308190,0.455766
Alectinib,0.606338,0.432755,0.482186,0.462174,0.051665,0.394424,0.370571,-0.053692,-0.663905,0.311545,...,-0.046801,0.133507,0.805088,0.433801,0.361481,-0.672015,-1.041705,0.932388,0.580151,1.133224
Alisertib,-0.138914,0.374145,0.471988,1.162445,0.166010,0.021290,0.163839,-0.394300,0.321040,0.274726,...,0.827108,0.451001,0.431238,-0.278617,-0.006660,0.935345,0.396854,0.949590,1.404944,1.498148
Amuvatinib,0.323960,0.024025,0.896973,0.137332,0.761684,0.700579,-0.005966,0.159919,-0.000135,1.477128,...,-0.710519,-0.008132,0.683871,0.562627,0.085676,-0.125773,-0.085908,0.404221,0.727253,0.847465
Apitolisib,-0.812716,-0.868754,0.235552,0.242402,-0.252183,1.115722,-0.695836,-0.241817,0.084656,0.238791,...,1.567446,0.960644,0.909867,-0.868273,-0.261560,-0.434047,-1.288164,3.798603,1.414472,0.522060
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Vinorelbine,-0.469232,-0.881484,1.631099,-0.827658,-0.607201,-0.268407,-0.090187,-1.069736,-1.514293,-0.891194,...,2.000113,0.277909,1.877651,0.023288,-0.893992,1.088035,-0.171316,2.534415,-0.330012,0.814545
Vismodegib,-0.128222,0.523229,0.437824,-0.636485,-0.956063,0.628203,-1.188343,0.012607,-1.343582,-0.372599,...,-1.175488,-1.751382,1.382196,-0.381074,-0.773983,-1.620968,1.522709,-0.667788,-0.215596,1.389532
Vorinostat,0.080033,-0.488793,0.731058,-0.908605,-0.015747,0.590572,0.576774,0.497313,0.046112,0.147323,...,-1.064512,-1.185163,2.034210,-0.450098,-0.017785,-1.253606,1.488853,-0.605094,1.618442,0.937129
Voxtalisib,0.680178,-0.865490,0.948322,-0.229252,0.046177,0.494849,0.173076,0.253633,-0.539577,0.034014,...,0.156357,0.994981,2.266868,-0.636065,0.919801,-1.187751,-0.504057,1.864152,0.403739,0.696189


# Masking

In [9]:
# Validation Mask
for _, row in X_val.iterrows():
    drug_response.loc[row["DRUG_NAME"], row["CELL_LINE_NAME"]] = 0

# Test Mask
for _, row in X_test.iterrows():
    drug_response.loc[row["DRUG_NAME"], row["CELL_LINE_NAME"]] = 0

# DTI

In [10]:
dti = pd.read_csv("../data/full_table.csv")
dti = dti.dropna(subset="Drug Name").reset_index(drop=True)
dti.head()

,Drug Name,DrugBank ID,PubChem CID,PubChem SID,SMILES,Gene,NSC
0,Lepirudin,DB00001,NaN,46507011.0,NaN,F2,NaN
1,Cetuximab,DB00002,NaN,46507042.0,NaN,EGFR,NaN
2,Cetuximab,DB00002,NaN,46507042.0,NaN,FCGR3B,NaN
3,Cetuximab,DB00002,NaN,46507042.0,NaN,C1QA,NaN
4,Cetuximab,DB00002,NaN,46507042.0,NaN,C1QB,NaN


In [11]:
dti = dti[dti["Drug Name"].isin(drug_response.index)]
dti.head()

,Drug Name,DrugBank ID,PubChem CID,PubChem SID,SMILES,Gene,NSC
1061,Bortezomib,DB00188,387447.0,46508736.0,CC(C)C[C@H](NC(=O)[C@H](CC1=CC=CC=C1)NC(=O)C1=...,PSMB5,681239.0
1062,Bortezomib,DB00188,387447.0,46508736.0,CC(C)C[C@H](NC(=O)[C@H](CC1=CC=CC=C1)NC(=O)C1=...,PSMB1,681239.0
1138,Pyrimethamine,DB00205,4993.0,46505987.0,CCC1=C(C(N)=NC(N)=N1)C1=CC=C(Cl)C=C1,DHFR,3061.0
1139,Pyrimethamine,DB00205,4993.0,46505987.0,CCC1=C(C(N)=NC(N)=N1)C1=CC=C(Cl)C=C1,DHFR,757306.0
1140,Pyrimethamine,DB00205,4993.0,46505987.0,CCC1=C(C(N)=NC(N)=N1)C1=CC=C(Cl)C=C1,HEXB,3061.0


In [12]:
print("unique drugs:", len(set(dti["Drug Name"])))
print("unique genes:", len(set(dti.Gene)))

unique drugs: 54
unique genes: 137


In [13]:
len(set(drug_response.index) & set(dti["Drug Name"]))

54

In [14]:
len(set(gene_exp.columns) & set(dti.Gene))

131

## All drugs are in drug response.

In [15]:
dti = dti[dti.Gene.isin(set(gene_exp.columns) & set(dti.Gene))]
dti

,Drug Name,DrugBank ID,PubChem CID,PubChem SID,SMILES,Gene,NSC
1061,Bortezomib,DB00188,387447.0,46508736.0,CC(C)C[C@H](NC(=O)[C@H](CC1=CC=CC=C1)NC(=O)C1=...,PSMB5,681239.0
1062,Bortezomib,DB00188,387447.0,46508736.0,CC(C)C[C@H](NC(=O)[C@H](CC1=CC=CC=C1)NC(=O)C1=...,PSMB1,681239.0
1140,Pyrimethamine,DB00205,4993.0,46505987.0,CCC1=C(C(N)=NC(N)=N1)C1=CC=C(Cl)C=C1,HEXB,3061.0
1141,Pyrimethamine,DB00205,4993.0,46505987.0,CCC1=C(C(N)=NC(N)=N1)C1=CC=C(Cl)C=C1,HEXB,757306.0
1555,Gefitinib,DB00317,123631.0,46508649.0,COC1=C(OCCCN2CCOCC2)C=C2C(NC3=CC(Cl)=C(F)C=C3)...,EGFR,715055.0
...,...,...,...,...,...,...,...
19475,Amuvatinib,DB12742,11282283.0,347828932.0,S=C(NCC1=CC=C2OCOC2=C1)N1CCN(CC1)C1=NC=NC2=C1O...,RET,NaN
19476,Amuvatinib,DB12742,11282283.0,347828932.0,S=C(NCC1=CC=C2OCOC2=C1)N1CCN(CC1)C1=NC=NC2=C1O...,PDGFRA,NaN
19477,Amuvatinib,DB12742,11282283.0,347828932.0,S=C(NCC1=CC=C2OCOC2=C1)N1CCN(CC1)C1=NC=NC2=C1O...,FLT3,NaN
19478,Amuvatinib,DB12742,11282283.0,347828932.0,S=C(NCC1=CC=C2OCOC2=C1)N1CCN(CC1)C1=NC=NC2=C1O...,RAD51,NaN


# Selected highly variable genes

In [16]:
variance = gene_exp.std()
variance = variance.sort_values(ascending=False)
variance = pd.DataFrame(variance > np.percentile(variance, 90))
variance = list(variance[variance[0] == True].index)
len(variance)

1742

In [17]:
print("DTI unique genes: ", len(set(dti["Gene"])))
print("Top 90% variable genes: ", len(variance))
print("Total: ", len(set(variance) | (set(dti["Gene"]))))

DTI unique genes:  131
Top 90% variable genes:  1742
Total:  1856


# Preprocessed data dims

In [18]:
genes = sorted(list(set(variance) | (set(dti["Gene"]))))
gene_exp = gene_exp[genes]
gene_exp.shape

(968, 1856)

In [19]:
drug_response.shape

(74, 904)

# Normalize

In [20]:
gene_norm_cell = pd.DataFrame(
    StandardScaler().fit_transform(gene_exp),
    index=gene_exp.index,
    columns=gene_exp.columns,
)
gene_norm_cell

,A2M,AAED1,ABCB1,ABCC3,ABCG1,ABI3BP,ABL1,ABL2,ABLIM1,ACAP1,...,ZNF300,ZNF415,ZNF462,ZNF467,ZNF608,ZNF711,ZNF83,ZNF853,ZP3,ZSCAN31
MC-CAR,-0.183823,0.315812,1.375685,-0.831891,-0.703865,-0.519713,-1.053562,-1.144418,-1.290349,2.284174,...,-1.278763,-0.758898,0.842565,-0.882461,-0.074280,-0.214688,-0.372614,-0.763770,-1.581768,-0.893358
PFSK-1,-0.209492,0.322895,-0.328531,-0.788525,1.602383,1.927808,0.850178,-0.843973,-0.880427,-0.486080,...,0.461066,-0.816545,-0.557179,-0.623753,0.331575,-0.962710,-0.090130,-0.176769,-1.074304,0.326869
A673,-0.373586,-1.387253,-0.660569,-0.863068,-0.279611,-0.342609,1.342116,-0.454606,-0.489952,-0.444870,...,0.444578,-0.398170,0.502540,1.333584,-0.911795,1.433794,0.188239,1.216433,0.735789,-0.847409
ES3,-0.270837,-0.723571,-0.444008,-0.996405,-0.280151,-0.031850,1.091420,0.283743,-0.975739,0.279643,...,1.498397,1.513859,0.726998,2.211373,-0.509698,1.700800,0.425042,1.264732,1.880398,-0.254039
ES5,1.647918,-0.659135,-0.559496,-0.942793,-0.806008,0.147963,0.280821,0.802122,-0.342000,-0.073761,...,0.963448,1.822567,-1.881274,2.290713,-0.374365,1.465508,0.824499,1.700572,-1.072722,-0.715958
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SNU-283,-0.196266,-1.162314,1.213693,1.933026,1.533893,-0.535923,0.664679,-1.499521,1.313865,-0.113262,...,-0.930435,-0.121002,0.268203,-0.501986,-0.340202,-0.819995,-0.163018,-0.847269,0.115015,-0.415638
SNU-407,-0.162459,-1.299035,0.705396,0.063787,-0.086131,-0.538136,-0.131771,-1.295700,1.357186,-0.552781,...,-1.048848,-0.974241,-0.565825,-0.880683,-0.611274,-0.141052,-2.517792,-0.596205,0.010876,-0.309140
SNU-61,-0.357438,-0.744172,1.009094,1.253740,-0.705407,-0.518680,0.580381,-1.007094,1.098923,-0.548713,...,-1.225661,-0.756175,0.586308,-0.980272,-1.008351,0.427593,0.079123,-0.653641,1.074412,-0.547522
SNU-81,-0.340462,-0.401068,-0.403490,0.933954,-0.671707,-0.479090,-0.652520,-1.699752,1.787090,-0.452524,...,-1.109103,-0.812905,0.275427,-0.308895,-1.042723,0.030764,-1.430484,-0.815159,0.549280,-0.035066


In [21]:
gene_norm_gene = pd.DataFrame(
    StandardScaler().fit_transform(gene_exp.T.values),
    index=gene_exp.columns,
    columns=gene_exp.index,
).T
gene_norm_gene

,A2M,AAED1,ABCB1,ABCC3,ABCG1,ABI3BP,ABL1,ABL2,ABLIM1,ACAP1,...,ZNF300,ZNF415,ZNF462,ZNF467,ZNF608,ZNF711,ZNF83,ZNF853,ZP3,ZSCAN31
MC-CAR,-0.601455,0.854352,0.377667,-0.809824,-0.765587,-0.937542,-0.103302,-0.201758,-0.333545,0.747630,...,-0.881550,-0.868577,1.139973,-0.635446,-0.207944,-0.448527,0.750616,-0.847331,-0.760614,-0.798985
PFSK-1,-0.681606,0.792876,-0.831947,-0.838759,0.581155,0.691736,0.362793,-0.173195,-0.099139,-0.945536,...,0.182237,-0.967182,0.122671,-0.514689,-0.025491,-1.040757,0.908511,-0.545054,-0.519396,-0.150396
A673,-0.792316,-0.194040,-1.041295,-0.882198,-0.561633,-0.864728,0.487882,-0.057716,0.176889,-0.909683,...,0.165366,-0.683041,0.825406,0.844600,-0.766555,0.644571,1.106566,0.307626,0.545544,-0.822822
ES3,-0.688454,0.244671,-0.874875,-0.957937,-0.526642,-0.621684,0.489935,0.223704,-0.119311,-0.453016,...,0.920719,0.634487,1.065927,1.559591,-0.491330,0.916163,1.392424,0.404597,1.314910,-0.445393
ES5,0.754047,0.266229,-1.035065,-0.997114,-0.929526,-0.552258,0.242989,0.372751,0.334881,-0.730574,...,0.567118,0.855201,-0.809504,1.670004,-0.460613,0.754092,1.772629,0.683593,-0.532415,-0.787797
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SNU-283,-0.676336,-0.092869,0.164968,1.077519,0.486257,-0.999958,0.267328,-0.388338,1.396820,-0.728695,...,-0.719583,-0.518912,0.624506,-0.443230,-0.444999,-0.933230,0.785317,-0.952427,0.149552,-0.590068
SNU-407,-0.702316,-0.208397,-0.202644,-0.278305,-0.508653,-1.058403,0.019752,-0.372177,1.419960,-1.033065,...,-0.845712,-1.117882,0.045999,-0.749648,-0.650755,-0.517145,-1.057099,-0.856357,0.055567,-0.577924
SNU-61,-0.884032,0.021139,-0.081910,0.470648,-0.918850,-1.077729,0.124635,-0.353004,1.102160,-1.063857,...,-0.994051,-1.012372,0.696946,-0.860929,-0.925133,-0.195981,0.830490,-0.930531,0.566090,-0.761020
SNU-81,-0.842224,0.282879,-0.944331,0.329860,-0.870621,-1.029730,-0.136610,-0.505369,1.702948,-0.986771,...,-0.895621,-1.024794,0.588444,-0.369419,-0.917489,-0.411388,-0.229970,-1.000319,0.353015,-0.435364


# Make matrices association matrices by setting 0 threshold and min max scaler.

In [22]:
def min_max_scale(data):
    scaler = MinMaxScaler(feature_range=(0, 1))
    data = data[data > 0].fillna(0)
    return pd.DataFrame(
        scaler.fit_transform(data), index=data.index, columns=data.columns
    )

In [23]:
A_dc = min_max_scale(drug_response)
A_dc

,22RV1,23132-87,42-MG-BA,451Lu,639-V,647-V,769-P,786-0,8-MG-BA,8305C,...,WSU-DLCL2,WSU-NHL,YAPC,YH-13,YKG-1,YT,ZR-75-30,huH-1,no-10,no-11
Afatinib,0.376393,0.517643,0.357774,0.425107,0.000000,0.000000,0.000000,0.000000,0.232496,0.185514,...,0.164813,0.247570,0.000000,0.112916,0.032795,0.247930,0.000000,0.281846,0.168034,0.156310
Alectinib,0.351775,0.449375,0.270324,0.312272,0.035323,0.293742,0.275463,0.000000,0.000000,0.100073,...,0.000000,0.087563,0.257096,0.312334,0.306472,0.000000,0.000000,0.289291,0.316315,0.388653
Alisertib,0.000000,0.388514,0.264607,0.785415,0.113499,0.015855,0.121789,0.000000,0.305698,0.088246,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.572744,0.137655,0.000000,0.766017,0.513808
Amuvatinib,0.187950,0.024948,0.502863,0.092789,0.520756,0.521747,0.000000,0.101821,0.000000,0.474475,...,0.000000,0.000000,0.000000,0.405088,0.072638,0.000000,0.000000,0.000000,0.396520,0.290648
Apitolisib,0.000000,0.000000,0.132056,0.163781,0.000000,0.830919,0.000000,0.000000,0.080610,0.076703,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.771211,0.179047
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Vinorelbine,0.000000,0.000000,0.914431,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.000000,0.182273,0.599607,0.016767,0.000000,0.666242,0.000000,0.786350,0.000000,0.279358
Vismodegib,0.000000,0.543323,0.000000,0.000000,0.000000,0.467846,0.000000,0.008027,0.000000,0.000000,...,0.000000,0.000000,0.441389,0.000000,0.000000,0.000000,0.528174,0.000000,0.000000,0.000000
Vorinostat,0.046432,0.000000,0.409848,0.000000,0.000000,0.439821,0.428744,0.316642,0.043908,0.047322,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.516431,0.000000,0.882422,0.321400
Voxtalisib,0.394614,0.000000,0.000000,0.000000,0.031571,0.368532,0.000000,0.161490,0.000000,0.010926,...,0.078174,0.652580,0.723899,0.000000,0.000000,0.000000,0.000000,0.578388,0.220130,0.000000


In [24]:
A_cg = min_max_scale(gene_norm_gene + gene_norm_cell)
A_cg

,A2M,AAED1,ABCB1,ABCC3,ABCG1,ABI3BP,ABL1,ABL2,ABLIM1,ACAP1,...,ZNF300,ZNF415,ZNF462,ZNF467,ZNF608,ZNF711,ZNF83,ZNF853,ZP3,ZSCAN31
MC-CAR,0.000000,0.301107,0.240791,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.521277,...,0.000000,0.000000,0.509502,0.000000,0.000000,0.000000,0.093779,0.000000,0.000000,0.000000
PFSK-1,0.000000,0.287111,0.000000,0.000000,0.462003,0.328815,0.271666,0.000000,0.000000,0.000000,...,0.148289,0.000000,0.000000,0.000000,0.061077,0.000000,0.203034,0.000000,0.000000,0.030406
A673,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.409860,0.000000,0.000000,0.000000,...,0.140599,0.000000,0.341275,0.417312,0.000000,0.470393,0.321231,0.324741,0.225373,0.000000
ES3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.354172,0.105840,0.000000,0.000000,...,0.557635,0.605506,0.460772,0.722468,0.000000,0.592293,0.450899,0.355694,0.562021,0.000000
ES5,0.271144,0.000000,0.000000,0.000000,0.000000,0.000000,0.117316,0.245047,0.000000,0.000000,...,0.352814,0.754722,0.000000,0.758822,0.000000,0.502358,0.644327,0.508009,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SNU-283,0.000000,0.000000,0.189334,0.748059,0.427433,0.000000,0.208739,0.000000,0.459842,0.000000,...,0.000000,0.000000,0.229422,0.000000,0.000000,0.000000,0.154388,0.000000,0.046534,0.000000
SNU-407,0.000000,0.000000,0.069044,0.000000,0.000000,0.000000,0.000000,0.000000,0.471116,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.011687,0.000000
SNU-61,0.000000,0.000000,0.127332,0.428475,0.000000,0.000000,0.157901,0.000000,0.373393,0.000000,...,0.000000,0.000000,0.329790,0.000000,0.000000,0.052420,0.225668,0.000000,0.288547,0.000000
SNU-81,0.000000,0.000000,0.000000,0.314032,0.000000,0.000000,0.000000,0.000000,0.592052,0.000000,...,0.000000,0.000000,0.222010,0.000000,0.000000,0.000000,0.000000,0.000000,0.158704,0.000000


In [25]:
A_dg = (
    pd.DataFrame(
        np.ones([len(A_dc.index), len(A_cg.columns)]),
        index=A_dc.index,
        columns=A_cg.columns,
    )
    / 2
)
for _, i in dti.iterrows():
    A_dg.loc[i["Drug Name"], i["Gene"]] = 1
A_dg

,A2M,AAED1,ABCB1,ABCC3,ABCG1,ABI3BP,ABL1,ABL2,ABLIM1,ACAP1,...,ZNF300,ZNF415,ZNF462,ZNF467,ZNF608,ZNF711,ZNF83,ZNF853,ZP3,ZSCAN31
Afatinib,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
Alectinib,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
Alisertib,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
Amuvatinib,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
Apitolisib,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Vinorelbine,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
Vismodegib,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
Vorinostat,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
Voxtalisib,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5


In [26]:
A_dg.sum().sum()

68782.0

In [27]:
print(
    "Drug Density: ",
    round(len(A_dc.values.nonzero()[0]) / drug_response.size, 4) * 100,
    "%",
)
print("Cell Density: ", round(len(A_cg.values.nonzero()[0]) / A_cg.size, 4) * 100, "%")
print("Gene Density: ", round(len(A_dg.values.nonzero()[0]) / A_dg.size, 5) * 100, "%")

Drug Density:  38.03 %
Cell Density:  41.54 %
Gene Density:  100.0 %


# Similarity

In [28]:
def normalize_similarity_matrix(df, gamma=None):
    similarity_matrix = rbf_kernel(df.values, gamma=gamma)
    scaler = MinMaxScaler()
    normalized_matrix = scaler.fit_transform(similarity_matrix.reshape(-1, 1))
    normalized_df = pd.DataFrame(
        normalized_matrix.reshape(similarity_matrix.shape),
        index=df.index,
        columns=df.index,
    )

    return normalized_df

In [29]:
cell_sim = normalize_similarity_matrix(drug_response.T)
cell_sim.to_csv("../GDSC1_data/cell_sim.csv")
cell_sim

,22RV1,23132-87,42-MG-BA,451Lu,639-V,647-V,769-P,786-0,8-MG-BA,8305C,...,WSU-DLCL2,WSU-NHL,YAPC,YH-13,YKG-1,YT,ZR-75-30,huH-1,no-10,no-11
22RV1,1.000000,0.601058,0.381711,0.322858,0.508466,0.503181,0.406953,0.367654,0.388796,0.426597,...,0.319457,0.385305,0.392481,0.421671,0.602004,0.184544,0.215685,0.336117,0.358183,0.404048
23132-87,0.601058,1.000000,0.481089,0.342138,0.477479,0.482644,0.424256,0.354007,0.467960,0.371646,...,0.390896,0.451290,0.281700,0.496770,0.633987,0.214131,0.203812,0.369068,0.295835,0.379888
42-MG-BA,0.381711,0.481089,1.000000,0.249357,0.456995,0.441720,0.379223,0.317690,0.469555,0.288475,...,0.461594,0.422685,0.205082,0.553492,0.503381,0.206596,0.108089,0.280571,0.235187,0.369452
451Lu,0.322858,0.342138,0.249357,1.000000,0.424466,0.384522,0.309170,0.240248,0.281036,0.547781,...,0.197801,0.209534,0.226038,0.306461,0.430297,0.102226,0.105964,0.293127,0.297764,0.308063
639-V,0.508466,0.477479,0.456995,0.424466,1.000000,0.586377,0.465732,0.497118,0.494311,0.477225,...,0.306514,0.330254,0.268169,0.480763,0.603087,0.195889,0.142097,0.387279,0.312685,0.430604
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
YT,0.184544,0.214131,0.206596,0.102226,0.195889,0.130583,0.244635,0.317085,0.180491,0.099124,...,0.223613,0.301118,0.083506,0.233299,0.239244,1.000000,0.070143,0.108815,0.072279,0.145440
ZR-75-30,0.215685,0.203812,0.108089,0.105964,0.142097,0.213319,0.125572,0.086635,0.112484,0.134056,...,0.120959,0.079938,0.237873,0.119415,0.172502,0.070143,1.000000,0.152772,0.214096,0.261248
huH-1,0.336117,0.369068,0.280571,0.293127,0.387279,0.401890,0.235286,0.163496,0.209022,0.367438,...,0.389706,0.225972,0.375059,0.273415,0.335538,0.108815,0.152772,1.000000,0.274695,0.383200
no-10,0.358183,0.295835,0.235187,0.297764,0.312685,0.411533,0.228728,0.161701,0.317267,0.351980,...,0.205530,0.143564,0.344191,0.162682,0.361708,0.072279,0.214096,0.274695,1.000000,0.546399


In [30]:
print("Min:", np.min(cell_sim.values))
print("Max:", np.max(cell_sim.values))
print("Mean:", np.mean(cell_sim.values))
print("Median:", np.median(cell_sim.values))

Min: 0.0
Max: 0.9999999999999999
Mean: 0.2923212017011527
Median: 0.2868609431734003


In [31]:
gene_sim = normalize_similarity_matrix(gene_norm_cell.T)
gene_sim.to_csv("../GDSC1_data/gene_sim.csv")
gene_sim

,A2M,AAED1,ABCB1,ABCC3,ABCG1,ABI3BP,ABL1,ABL2,ABLIM1,ACAP1,...,ZNF300,ZNF415,ZNF462,ZNF467,ZNF608,ZNF711,ZNF83,ZNF853,ZP3,ZSCAN31
A2M,1.000000,0.122510,0.123400,0.088330,0.075220,0.118393,0.116858,0.175571,0.084031,0.089788,...,0.125916,0.119805,0.141814,0.092885,0.157358,0.085962,0.099398,0.130532,0.080746,0.089855
AAED1,0.122510,1.000000,0.103315,0.162484,0.065855,0.187494,0.177476,0.226015,0.082277,0.045829,...,0.158549,0.119975,0.210392,0.114317,0.078720,0.059882,0.100786,0.100737,0.169264,0.154126
ABCB1,0.123400,0.103315,1.000000,0.160006,0.100523,0.112892,0.114793,0.104328,0.167157,0.114300,...,0.118313,0.093226,0.116010,0.097047,0.154011,0.131755,0.093676,0.113404,0.101951,0.105067
ABCC3,0.088330,0.162484,0.160006,1.000000,0.136852,0.155008,0.109648,0.112810,0.246798,0.046345,...,0.098182,0.099522,0.151761,0.124760,0.093997,0.046710,0.099932,0.083201,0.193903,0.226953
ABCG1,0.075220,0.065855,0.100523,0.136852,1.000000,0.082214,0.095874,0.056406,0.188355,0.138101,...,0.095143,0.104194,0.080687,0.167873,0.096808,0.163991,0.113069,0.129339,0.122641,0.149008
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZNF711,0.085962,0.059882,0.131755,0.046710,0.163991,0.075437,0.147640,0.070592,0.116553,0.135689,...,0.202422,0.170409,0.133474,0.152286,0.156069,1.000000,0.141010,0.265314,0.079753,0.106820
ZNF83,0.099398,0.100786,0.093676,0.099932,0.113069,0.149690,0.160597,0.125930,0.087022,0.124461,...,0.156094,0.311728,0.120299,0.163820,0.168165,0.141010,1.000000,0.157693,0.097302,0.111617
ZNF853,0.130532,0.100737,0.113404,0.083201,0.129339,0.097803,0.156404,0.115781,0.121869,0.063756,...,0.212188,0.223951,0.194026,0.176827,0.177194,0.265314,0.157693,1.000000,0.083953,0.123620
ZP3,0.080746,0.169264,0.101951,0.193903,0.122641,0.121772,0.102588,0.131102,0.165964,0.072506,...,0.093924,0.099423,0.122103,0.192523,0.074307,0.079753,0.097302,0.083953,1.000000,0.146863


In [32]:
print("Min:", np.min(gene_sim.values))
print("Max:", np.max(gene_sim.values))
print("Mean:", np.mean(gene_sim.values))
print("Median:", np.median(gene_sim.values))

Min: 0.0
Max: 1.0
Mean: 0.13403773400046046
Median: 0.11951040014733928


# NSC to SMILES

In [33]:
convert = dict(
    pd.read_csv("../Figs/nsc_cid_smiles_class_name.csv", index_col=0)[
        ["NAME", "SMILES"]
    ].values
)
SMILES = [convert[i] for i in drug_response.index]
SMILES

['CN(C)CC=CC(=O)NC1=C(C=C2C(=C1)C(=NC=N2)NC3=CC(=C(C=C3)F)Cl)OC4CCOC4',
 'CCC1=CC2=C(C=C1N3CCC(CC3)N4CCOCC4)C(C5=C(C2=O)C6=C(N5)C=C(C=C6)C#N)(C)C',
 'COC1=C(C(=CC=C1)F)C2=NCC3=CN=C(N=C3C4=C2C=C(C=C4)Cl)NC5=CC(=C(C=C5)C(=O)O)OC',
 'C1CN(CCN1C2=NC=NC3=C2OC4=CC=CC=C43)C(=S)NCC5=CC6=C(C=C5)OCO6',
 'CC1=C(SC2=C1N=C(N=C2N3CCOCC3)C4=CN=C(N=C4)N)CN5CCN(CC5)C(=O)C(C)O',
 'C1=CC(=CC=C1S(=O)(=O)N(CC2=C(C=C(C=C2)C3=NOC=N3)F)C(CCC(F)(F)F)C(=O)N)Cl',
 'CNC(=O)C1=CC=CC=C1SC2=CC3=C(C=C2)C(=NN3)C=CC4=CC=CC=N4',
 'C1=CC=C(C=C1)NS(=O)(=O)C2=CC=CC(=C2)C=CC(=O)NO',
 'B(C(CC(C)C)NC(=O)C(CC1=CC=CC=C1)NC(=O)C2=NC=CN=C2)(O)O',
 'CN1CCN(CC1)CCCOC2=C(C=C3C(=C2)N=CC(=C3NC4=CC(=C(C=C4Cl)Cl)OC)C#N)OC',
 'COC1=CC2=C(C=CN=C2C=C1OC)OC3=CC=C(C=C3)NC(=O)C4(CC4)C(=O)NC5=CC=C(C=C5)F',
 'C1CN(CCC1(C(=O)NC(CCO)C2=CC=C(C=C2)Cl)N)C3=NC=NC4=C3C=CN4',
 'CC(C1=C(C=CC(=C1Cl)F)Cl)OC2=C(N=CC(=C2)C3=CN(N=C3)C4CCNCC4)N',
 'CC1CC2C(C(C3(O2)CCC4C5CC=C6CC(CCC6(C5CC4=C3C)C)O)C)NC1',
 'CC(C)(C)C1=NC(=C(S1)C2=NC(=NC=C2)N)C3=C(C(=CC=C3)NS(=

In [34]:
def get_morgan_fingerprint(SMILES):
    # Initialize parser parameters
    params = Chem.SmilesParserParams()
    params.useChirality = True  # Preserve stereochemistry information
    params.removeHs = False  # Keep hydrogen atoms
    mfp = []

    for smi in SMILES:
        mol = None
        # Sanitization attempt strategies
        sanitize_attempts = [
            {"sanitize": True},  # First try with standard sanitization
            {
                "sanitize": False,
                "partial_sanitize": True,
            },  # Fallback: partial sanitization
        ]

        for attempt in sanitize_attempts:
            try:
                # Update parameters for this attempt
                current_params = Chem.SmilesParserParams()
                current_params.sanitize = attempt["sanitize"]
                current_params.useChirality = params.useChirality
                current_params.removeHs = params.removeHs

                # Molecule object creation
                mol = Chem.MolFromSmiles(smi, params=current_params)

                if mol and "partial_sanitize" in attempt:
                    # Perform partial sanitization (skip property validation)
                    Chem.SanitizeMol(mol, Chem.SANITIZE_ALL ^ Chem.SANITIZE_PROPERTIES)

                if mol:  # Successfully processed molecule
                    # Generate Morgan fingerprint
                    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
                    mfp.append(np.array(fp))
                    break  # Exit attempt loop on success

            except Exception as e:
                if attempt == sanitize_attempts[-1]:  # Final attempt failed
                    print(f"Processing failed: {smi}")
                    print(f"Error details: {str(e)}")
                continue  # Try next attempt

        if not mol:  # All attempts failed
            print(f"Complete processing failure: {smi}")
            mfp.append(np.zeros(2048))  # Insert zero-vector placeholder

    return np.array(mfp)

In [35]:
mfp = get_morgan_fingerprint(SMILES)
mfp = pd.DataFrame(mfp, index=drug_response.index)

In [36]:
drug_sim = normalize_similarity_matrix(mfp)
drug_sim.to_csv("../GDSC1_data/drug_sim.csv")
drug_sim

,Afatinib,Alectinib,Alisertib,Amuvatinib,Apitolisib,Avagacestat,Axitinib,Belinostat,Bortezomib,Bosutinib,...,Tivozanib,Tozasertib,Trametinib,Tretinoin,Veliparib,Vinorelbine,Vismodegib,Vorinostat,Voxtalisib,Z-LLNle-CHO
Afatinib,1.000000,0.275408,0.392423,0.307809,0.275408,0.333787,0.346795,0.412025,0.320791,0.385895,...,0.451317,0.333787,0.320791,0.353303,0.372849,0.082654,0.425110,0.457877,0.346795,0.340289
Alectinib,0.275408,1.000000,0.256005,0.353303,0.346795,0.288359,0.353303,0.392423,0.314298,0.340289,...,0.301322,0.353303,0.288359,0.398954,0.457877,0.230179,0.366330,0.451317,0.444760,0.333787
Alisertib,0.392423,0.256005,1.000000,0.301322,0.307809,0.379370,0.379370,0.457877,0.431657,0.392423,...,0.418566,0.366330,0.353303,0.385895,0.444760,0.140184,0.484148,0.503885,0.392423,0.385895
Amuvatinib,0.307809,0.353303,0.301322,1.000000,0.405488,0.346795,0.412025,0.464440,0.425110,0.359815,...,0.372849,0.385895,0.294839,0.418566,0.451317,0.197967,0.438207,0.536845,0.438207,0.405488
Apitolisib,0.275408,0.346795,0.307809,0.405488,1.000000,0.353303,0.353303,0.392423,0.418566,0.340289,...,0.340289,0.379370,0.275408,0.425110,0.444760,0.101802,0.379370,0.477575,0.484148,0.385895
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Vinorelbine,0.082654,0.230179,0.140184,0.197967,0.101802,0.095416,0.210842,0.249544,0.185104,0.185104,...,0.197967,0.172254,0.133779,0.230179,0.314298,1.000000,0.197967,0.320791,0.236631,0.217285
Vismodegib,0.425110,0.366330,0.484148,0.438207,0.379370,0.464440,0.569885,0.609639,0.530246,0.412025,...,0.517059,0.464440,0.412025,0.484148,0.530246,0.197967,1.000000,0.642857,0.517059,0.484148
Vorinostat,0.457877,0.451317,0.503885,0.536845,0.477575,0.497303,0.589747,0.749702,0.629561,0.484148,...,0.550051,0.550051,0.523651,0.622917,0.629561,0.320791,0.642857,1.000000,0.616277,0.662827
Voxtalisib,0.346795,0.444760,0.392423,0.438207,0.484148,0.412025,0.530246,0.530246,0.490724,0.438207,...,0.477575,0.490724,0.412025,0.510470,0.556659,0.236631,0.517059,0.616277,1.000000,0.497303


In [37]:
print("Min:", np.min(drug_sim.values))
print("Max:", np.max(drug_sim.values))
print("Mean:", np.mean(drug_sim.values))
print("Median:", np.median(drug_sim.values))

Min: 0.0
Max: 1.0
Mean: 0.3919569257045593
Median: 0.38589474012676916


# Unified Graph

In [38]:
A_cg

,A2M,AAED1,ABCB1,ABCC3,ABCG1,ABI3BP,ABL1,ABL2,ABLIM1,ACAP1,...,ZNF300,ZNF415,ZNF462,ZNF467,ZNF608,ZNF711,ZNF83,ZNF853,ZP3,ZSCAN31
MC-CAR,0.000000,0.301107,0.240791,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.521277,...,0.000000,0.000000,0.509502,0.000000,0.000000,0.000000,0.093779,0.000000,0.000000,0.000000
PFSK-1,0.000000,0.287111,0.000000,0.000000,0.462003,0.328815,0.271666,0.000000,0.000000,0.000000,...,0.148289,0.000000,0.000000,0.000000,0.061077,0.000000,0.203034,0.000000,0.000000,0.030406
A673,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.409860,0.000000,0.000000,0.000000,...,0.140599,0.000000,0.341275,0.417312,0.000000,0.470393,0.321231,0.324741,0.225373,0.000000
ES3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.354172,0.105840,0.000000,0.000000,...,0.557635,0.605506,0.460772,0.722468,0.000000,0.592293,0.450899,0.355694,0.562021,0.000000
ES5,0.271144,0.000000,0.000000,0.000000,0.000000,0.000000,0.117316,0.245047,0.000000,0.000000,...,0.352814,0.754722,0.000000,0.758822,0.000000,0.502358,0.644327,0.508009,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SNU-283,0.000000,0.000000,0.189334,0.748059,0.427433,0.000000,0.208739,0.000000,0.459842,0.000000,...,0.000000,0.000000,0.229422,0.000000,0.000000,0.000000,0.154388,0.000000,0.046534,0.000000
SNU-407,0.000000,0.000000,0.069044,0.000000,0.000000,0.000000,0.000000,0.000000,0.471116,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.011687,0.000000
SNU-61,0.000000,0.000000,0.127332,0.428475,0.000000,0.000000,0.157901,0.000000,0.373393,0.000000,...,0.000000,0.000000,0.329790,0.000000,0.000000,0.052420,0.225668,0.000000,0.288547,0.000000
SNU-81,0.000000,0.000000,0.000000,0.314032,0.000000,0.000000,0.000000,0.000000,0.592052,0.000000,...,0.000000,0.000000,0.222010,0.000000,0.000000,0.000000,0.000000,0.000000,0.158704,0.000000


In [39]:
indexes = list(A_dc.index) + list(A_cg.index) + list(A_dg.columns)
n_all = len(indexes)
base = pd.DataFrame(np.zeros([n_all, n_all]), index=indexes, columns=indexes)
base

,Afatinib,Alectinib,Alisertib,Amuvatinib,Apitolisib,Avagacestat,Axitinib,Belinostat,Bortezomib,Bosutinib,...,ZNF300,ZNF415,ZNF462,ZNF467,ZNF608,ZNF711,ZNF83,ZNF853,ZP3,ZSCAN31
Afatinib,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Alectinib,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Alisertib,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Amuvatinib,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Apitolisib,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZNF711,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ZNF83,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ZNF853,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ZP3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [40]:
base.loc[A_cg.index, A_cg.columns] = A_cg
base.loc[A_cg.columns, A_cg.index] = A_cg.T
base.loc[A_dc.index, A_dc.columns] = A_dc
base.loc[A_dc.columns, A_dc.index] = A_dc.T
base.loc[A_dg.index, A_dg.columns] = A_dg
base.loc[A_dg.columns, A_dg.index] = A_dg.T

In [41]:
base

,Afatinib,Alectinib,Alisertib,Amuvatinib,Apitolisib,Avagacestat,Axitinib,Belinostat,Bortezomib,Bosutinib,...,ZNF300,ZNF415,ZNF462,ZNF467,ZNF608,ZNF711,ZNF83,ZNF853,ZP3,ZSCAN31
Afatinib,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
Alectinib,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
Alisertib,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
Amuvatinib,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
Apitolisib,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZNF711,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ZNF83,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ZNF853,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ZP3,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [42]:
edge_index = np.array(base.values.nonzero())
edge_attr = np.array(base.values[base.values.nonzero()])

np.save(
    "../GDSC1_data/idxs.npy",
    pd.DataFrame([list(range(len(base.index))), base.index]).values,
)

np.save(
    "../GDSC1_data/edge_idxs.npy",
    edge_index,
)

np.save(
    "../GDSC1_data/edge_attr.npy",
    edge_attr,
)